# 01 — Setup | Metadata and Quality Tables

**Project:** MLB Performance Intelligence Platform

Operational tables shared by every notebook. Bronze, Silver and Gold tables are created by
their own notebooks on first write, so their schema lives next to the logic that fills them.

| Table | Grain | Written by |
|---|---|---|
| `metadata.pipeline_runs` | one row per task and run (status, records, duration, error) | every notebook through `pipeline.log_step` |
| `metadata.extract_register` | one row per API request (URL, HTTP status, SHA-256, landed file) | `bronze/01_MLB_API_Extract` |
| `metadata.watermarks` | one row per incremental consumer (last Bronze ingestion processed) | Silver notebooks |
| `quality.dq_results` | one row per check and run | `quality/01_Data_Quality_Checks` |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
importlib.reload(C)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {C.PIPELINE_RUNS} (
  pipeline_name STRING, run_id STRING, layer STRING, task STRING,
  status STRING COMMENT 'SUCCESS or FAILED',
  message STRING COMMENT 'OK, a short summary, or the error traceback',
  execution_window STRING COMMENT 'Dates or seasons processed by the task',
  records_processed BIGINT, started_at TIMESTAMP, ended_at TIMESTAMP, duration_seconds DOUBLE
) USING DELTA
COMMENT 'Run log: one row per pipeline task execution (UTC timestamps)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {C.EXTRACT_REGISTER} (
  endpoint STRING COMMENT 'schedule, boxscore, people, teams, venues or standings',
  request_key STRING COMMENT 'Date range, gamePk, season or batch label identifying the request',
  request_url STRING, status_code INT COMMENT 'HTTP status (-1 = network error after retries)',
  outcome STRING COMMENT 'LANDED (new file), UNCHANGED (same SHA-256 as last extraction, not landed) or FAILED',
  error STRING, payload_sha256 STRING, file_path STRING, bytes BIGINT,
  extracted_at TIMESTAMP, run_id STRING
) USING DELTA
COMMENT 'Extraction register: every MLB Stats API request with its outcome; drives incremental extraction'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {C.WATERMARKS} (
  consumer STRING, high_watermark TIMESTAMP COMMENT 'Last bronze _ingested_at processed', updated_at TIMESTAMP
) USING DELTA
COMMENT 'Incremental processing watermarks for Silver consumers'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {C.DQ_RESULTS} (
  run_id STRING, check_id STRING, check_name STRING, layer STRING, table_name STRING,
  severity STRING COMMENT 'CRITICAL fails the job; WARNING is reported only',
  status STRING COMMENT 'PASS or FAIL', failed_rows BIGINT, checked_rows BIGINT, details STRING,
  checked_at TIMESTAMP
) USING DELTA
COMMENT 'Data-quality results: one row per check and run'
""")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {C.CATALOG}.metadata"))